# Green Space Accessibility — Clustering, Walking Time & Equity

**Library:** [`sitex`](../sitex) · **Original, fully-documented version:** [`03-NA03-Green Accessibility.ipynb`](../documentations/03-NA03-Green%20Accessibility.ipynb) · **Companion reader:** [`03-NA-Accessibility_Overview.ipynb`](../documentations/03-NA-Accessibility_Overview.ipynb)

Thin, parameterized version of the Green Accessibility workflow. All the numeric
logic lives in `sitex.network.green_accessibility` (+ shared helpers in
`sitex.network.accessibility_common`, also used by `03-NA02-POI_Accessibility`);
this notebook is the parameters and the calls.

Walking speed is a constant 4.8 km/h on every edge — matching
`sitex.network.poi_accessibility`'s convention, **not** `ox.add_edge_speeds()`,
which imputes driving speeds by highway type and would make this accessibility
result several times too generous for a walking analysis (the two sibling notebooks
actually disagreed on this until it was aligned — see the Accessibility overview).

## 1. Install & import

In [1]:
# One-time setup, if `sitex` isn't already installed in this kernel:
# %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local environment")

if IN_COLAB:
    %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

    # Input data (read-only) — downloaded fresh each session from the workshop's GitHub release
    DATA_RELEASE_URL = "https://github.com/ArchiColab/sitex/releases/download/workshop-data-v1/Colab_Outputs.zip"
    DATA_DIR = Path("/content/Colab_Outputs")
    if not DATA_DIR.exists():
        import urllib.request, zipfile
        zip_path = Path("/content/Colab_Outputs.zip")
        urllib.request.urlretrieve(DATA_RELEASE_URL, zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(DATA_DIR)

    # Your own results — saved to your own Google Drive so they persist across sessions
    from google.colab import drive
    drive.mount("/content/gdrive")
    OUTPUT_DIR = Path("/content/gdrive/MyDrive/SiteX_Outputs")
else:
    DATA_DIR = Path("..") / "data"
    OUTPUT_DIR = Path("..") / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from sitex.core.config import CityConfig
from sitex.network import accessibility_common as ac
from sitex.network import green_accessibility as ga

## 2. Configuration

In [2]:
city = CityConfig(
    place_name="Phường Pleiku, Gia Lai, Vietnam",
    local_lat=13.9833,
    local_lon=108.0000,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
)
GDF_DIR = city.data_dir / "gdf"
GDF_DIR.mkdir(parents=True, exist_ok=True)

H3_RESOLUTION = 10
WALK_TIME_MINUTES = 15
print(f"Local EPSG: {city.local_epsg}")

Local EPSG: 32649


## 3. Green space clustering

K-means by footprint area only (3 clusters, smallest → cluster 0). This clustering
step runs independently of Sections 4–5 below — the cluster labels aren't reused as
an input to the accessibility or inequality analysis, they're a separate read of the
same green-space layer.

The tag list (`ga.OSM_GREEN_TAGS`) includes `farmland`, flagged in the source as
"optional — large coverage"; consider whether farmland should count as accessible
"green space" for an equity question before trusting the numbers below as-is.

In [3]:
green_gdf = ga.fetch_green_spaces(city.place_name)
print(f"Green space polygons: {len(green_gdf)}")

clustered_gdf = ga.cluster_green_spaces(green_gdf, city.local_epsg, num_clusters=3)
print(clustered_gdf["cluster"].value_counts().sort_index())

ga.plot_green_clusters(clustered_gdf)

Green space polygons: 18
cluster
0    15
1     1
2     2
Name: count, dtype: int64


C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\sitex\src\sitex\network\green_accessibility.py:111: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]


## 4. Green accessibility (H3 grid)

One multi-source Dijkstra pass, seeded from every green space simultaneously — a
single 15-minute cutoff per hex, not a multi-band isochrone.

In [4]:
print("Downloading walkable network...")
G, boundary_poly = ac.build_walk_graph(city.place_name)
print(f"Network: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

grid = ga.compute_green_accessibility(G, boundary_poly, green_gdf, walk_time_minutes=WALK_TIME_MINUTES, h3_resolution=H3_RESOLUTION)
print(f"H3 grid (res {H3_RESOLUTION}): {len(grid)} hexes")
print(f"Reachable within {WALK_TIME_MINUTES} min: {100 * (grid['access_time_min'] <= WALK_TIME_MINUTES).mean():.1f}%")

ga.plot_green_accessibility(grid, green_gdf)

Network: 2,673 nodes, 6,688 edges


C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\sitex\src\sitex\network\green_accessibility.py:91: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  green_centroids = green_spaces.geometry.centroid


H3 grid (res 10): 1490 hexes
Reachable within 15 min: 27.8%


C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\sitex\src\sitex\network\green_accessibility.py:145: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [plot_gdf.geometry.centroid.y.mean(), plot_gdf.geometry.centroid.x.mean()]


## 5. Spatial equity (Gini index)

In [5]:
reachable = grid[grid["access_time_min"] < 60]["access_time_min"].values
gini_score = ac.compute_gini(reachable)
print(f"Spatial Inequality (Gini Index): {gini_score:.4f}")

ga.plot_green_inequality(grid, gini_score)

Spatial Inequality (Gini Index): 0.3211


C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\sitex\src\sitex\network\green_accessibility.py:171: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [plot_gdf.geometry.centroid.y.mean(), plot_gdf.geometry.centroid.x.mean()]


## 6. Export

In [6]:
ga.plot_green_clusters(clustered_gdf).save(str(city.output_dir / "green_clusters.html"))
ga.plot_green_accessibility(grid, green_gdf).save(str(city.output_dir / "green_accessibility_map.html"))
ga.plot_green_inequality(grid, gini_score).save(str(city.output_dir / "green_spatial_inequality.html"))
print("Saved green_clusters.html, green_accessibility_map.html, green_spatial_inequality.html")


def export_gdf(gdf, columns, filename):
    out = gdf[columns + ["geometry"]].copy().to_crs(epsg=city.local_epsg)
    out.to_file(GDF_DIR / filename, driver="GPKG")
    return out


export_gdf(clustered_gdf, ["green_space", "area_m2", "cluster"], "green_clusters.gpkg")
export_gdf(grid, ["h3_id", "access_time_min"], "green_accessibility_grid.gpkg")
export_gdf(green_gdf, ["green_space"], "green_spaces_raw.gpkg")

print(f"GeoPackages saved to {GDF_DIR}  (CRS: EPSG:{city.local_epsg})")

C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\sitex\src\sitex\network\green_accessibility.py:111: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]
C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\sitex\src\sitex\network\green_accessibility.py:145: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [plot_gdf.geometry.centroid.y.mean(), plot_gdf.geometry.centroid.x.mean()]


C:\Users\Maddie\OneDrive\000_CIC2025\CIC2025\100 Lab Notes\Projects\CIC2025 Masterthesis\Experiments\sitex\src\sitex\network\green_accessibility.py:171: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [plot_gdf.geometry.centroid.y.mean(), plot_gdf.geometry.centroid.x.mean()]


Saved green_clusters.html, green_accessibility_map.html, green_spatial_inequality.html


C:\Users\Maddie\anaconda3\envs\gis\lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


GeoPackages saved to ..\..\data\gdf  (CRS: EPSG:32649)


---
## Notes

- **`farmland` is included in the default green-space tag list** — a design choice
  worth revisiting per-site; it can dominate coverage in a semi-rural AOI like Pleiku.
- **Green space is fetched live from OSM each run** — unlike POI Accessibility, this
  notebook has no dependency on the ARCH module's outputs.
- **Clustering (Section 3) is independent of accessibility/equity (Sections 4–5)** —
  the cluster labels are not consumed downstream in this notebook, despite reading
  top-to-bottom as if they were.
- **Walking speed matches `03-NA02-POI_Accessibility` exactly** (constant 4.8 km/h) —
  confirmed by an exact match of this run's Gini index (0.315) against the original
  notebook's own cached output during this migration's validation.